# 08's PPO on the MLX stack

`08` moved generation to MLX: mlx-lm samples the rollouts and evaluations, with the policy's LoRA folded into its weights before each rollout. The rest of a step stayed in PyTorch on MPS -- the update, the reference's log-probabilities, the separate critic's reads -- about 85% of it. This notebook is 08, copied, with all of it on MLX, and no PyTorch in the process (`--trainer mlx`):

| | 08 | 08.5 |
|---|---|---|
| rollouts and evaluations | MLX | MLX |
| the update (LoRA, PPO's actor loss, AdamW) | PyTorch, MPS | MLX |
| the reference's log-probabilities (KL to the seed) | PyTorch, MPS | MLX |
| the separate critic's reads (Qwen3-4B, layer 27) | PyTorch, MPS | MLX |
| masks, GAE by the turn, the prover's progress, the critic's closed-form heads | PyTorch | MLX (the heads float64, on MLX's CPU stream) |

What it took (`08.5_mlx_stack.py`, the section "08.5: the MLX stack"):

* **Qwen3.5's gated delta rule, differentiable.** mlx-lm's Metal kernel has no backward, and its training path is a per-token loop that keeps every token's recurrent state for the backward -- 16 × 128 × 128 float32 per token per layer, ~90 GB for one 5.4k-token episode. `mlx_chunk_gated_delta_rule` is 08's chunked MPS rule (64 tokens a chunk, float32 inside) in MLX ops, left to MLX's autograd.
* **Gradient checkpointing per decoder layer** (`mlx_checkpoint_layers`): without it the full-attention layers' T × T scores of every layer are kept for the backward (a 7.9k-token episode peaked at 55 GB); with it, under 10 GB.
* **LoRA in PEFT's format both ways** (`MLXTrainable`): mlx-lm's `LoRALinear` stores A and B transposed and defaults to scale 20; here lora_a = Aᵀ, lora_b = Bᵀ, scale lora_alpha / r = 2, and checkpoints are written as PEFT's, so `probe` and everything else read them as before.
* **AdamW as PyTorch's**: MLX's Adam has no bias correction unless asked.
* **The critic on mlx-lm** (`MLXSeparateCritic`): the residual stream after layer 27 at the last token before each turn, the system turn read once and its cache copied per episode, as 08's.
* **The bookkeeping without PyTorch** (`mx_batch`, `mx_gae_turn`, `mx_normalized_advantages`, `MXCriticHeads`, ...): 08's functions, the same computation over MLX arrays. `train_ppo_mlx` runs under `block_torch`. Transformers then reports "PyTorch was not found" and loads its tokenizers only, any `import torch` would raise, and the run ends by printing whether PyTorch was loaded (it was not).

One trap on the way, silent: in MLX 0.32.3 the backward of the strided slices `inv[..., 0::2]` / `inv[..., 1::2]` in the triangular inverse came out wrong -- forward right, gradient cosine 0.56 against float64. Taking each pair by a reshape fixed it.

**The run.** D37: 08's configuration with three of the 37 PPO implementation details (advantages normalized per minibatch, 4 minibatches × 4 epochs, gradient clip 0.5, Adam eps 1e-5, lr 2e-6 annealed linearly to 0), 200 steps. It starts from 07's seed and 07's first 25 steps (the same questions at every step), so it pairs step by step with 07's run and 08's first run (v1). It was first started in PyTorch and stopped at step 29, then started again here, on MLX from the start.

**Continued to convergence.** From step 200 the lr was annealed again, 2e-6 to 0 at step 400 (`LR_ANNEAL_START=200`, `STEPS=400`), with evaluations every 20 steps and the run stopped once 6 in a row did not beat the best: it stopped at step 380, the best at step 260. From step 265 its rollouts were decoded in batches (`GENERATOR=mlx-batch`, `rollout_mlx_batched`: each round's episodes decoded together by mlx-lm's `BatchGenerator`), its evaluations as before (`EVAL_GENERATOR=mlx`), so that they stay token for token comparable with the earlier ones.


In [1]:
import importlib.util, json, sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "08.5_mlx_stack.py").exists() else Path("experiments/notebooks/smoke_test35")
# This notebook's module (08's, copied and changed as its header says), loaded by path. Nothing below needs a GPU:
# every number is read from the records.
spec = importlib.util.spec_from_file_location("pc085", HERE / "08.5_mlx_stack.py")
pc = importlib.util.module_from_spec(spec)
sys.modules["pc085"] = pc
spec.loader.exec_module(pc)
RUNS, RUNS08, RUNS07 = pc.RUNS, HERE / "08_prover_critic_runs", pc.RUNS07
PPO07, PPO08_V1, PPO08_D37 = "ppo-n200-r100-turn-s0", "ppo-qwen3-4b-bo4-a0.8-s0", "ppo-qwen3-4b-bo4-a0.8-d37-s0"
PPO085 = "ppo-mlx-qwen3-4b-bo4-a0.8-d37-s0"
PPO085_PC = "ppo-mlx-policy-d37-s0"  # 07's critic (critic=policy) + D37's details, no prover, 400 steps


def rows_of(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def jload(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


def median(xs):
    xs = sorted(xs)
    return (xs[(len(xs) - 1) // 2] + xs[len(xs) // 2]) / 2 if xs else float("nan")

## Checks against 08's PyTorch pieces

`08.5_mlx_stack.py trainer-check`, on MPS and MLX in one process (`08.5_mlx_stack_trainer_checks.json`):

1. **The rule**, on one episode's sizes (T 5.4k, 16 heads of 128), in float32: the output and the gradient of every input, MLX's chunked rule against 08's MPS one, and the time of a forward + backward.
2. **The inverse's gradient** against `torch.linalg.inv` in float64 (the trap above).
3. **The policy**: 07's seed (the run's start), on recorded episodes of 08's first run spread over a step's lengths.
   - Each policy token's log-probability, PyTorch's against MLX's.
   - The gradient of their sum with respect to every LoRA tensor.
   - One episode's forward + backward time: PyTorch with gradient checkpointing, as `train_ppo` loads it, against MLX with its layers checkpointed.

   bf16 both ways, so the two differ at bf16's rounding.
4. **The critic's features**, per turn.

In [2]:
CHK = jload(HERE / "08.5_mlx_stack_trainer_checks.json")
if CHK is None:
    print("trainer-check not run yet")
else:
    r = CHK["delta_rule_float32"]
    print(f"mlx {CHK['mlx']}, torch {CHK['torch']}; the rule's chunk {CHK['chunk']}")
    print(f"1. the rule, T={r['tokens']}, float32: output cosine {r['output']['cosine']:.7f} (max |diff| {r['output']['max_abs_diff']:.1e})")
    for k in ("q", "k", "v", "g", "beta"):
        x = r[f"grad_{k}"]
        print(f"   gradient of {k:4}: cosine {x['cosine']:.7f}, max |diff| {x['max_abs_diff']:.1e} (largest {x['scale']:.2f})")
    s = r["seconds_forward_backward"]
    print(f"   forward + backward: PyTorch MPS {min(s['torch_mps']):.3f}s, MLX {min(s['mlx']):.3f}s (best of 3)")
    x = CHK["inverse_grad_vs_float64"]
    print(f"2. the inverse's gradient against float64: cosine {x['cosine']:.8f}, max |diff| {x['max_abs_diff']:.1e}")
    print("3. the policy (07's seed), recorded episodes:")
    for p in CHK["policy"]:
        s = p["seconds_forward_backward"]
        print(f"   {p['tokens']:5} tokens ({p['policy_tokens']:4} the policy's): log-prob sums {p['logprob_sum_torch']:8.2f} / "
              f"{p['logprob_sum_mlx']:8.2f}, per token mean |diff| {p['logprob_mean_abs_diff']:.4f} (max {p['logprob_max_abs_diff']:.3f}) | "
              f"gradient cosine median {p['grad_cosine_median']:.4f} (min {p['grad_cosine_min']:.4f}), norm ratio {p['grad_norm_ratio_median']:.3f} | "
              f"forward + backward {s['torch_mps']:.1f}s / {s['mlx']:.1f}s")
    print("4. the critic's features (Qwen3-4B, layer 27):")
    for c in CHK["critic"]:
        print(f"   {c['turns']} turns: cosine per turn mean {c['cosine_mean']:.5f} (min {c['cosine_min']:.5f}), norm ratio {c['norm_ratio']:.4f}")

mlx 0.32.3, torch 2.14.1; the rule's chunk 64
1. the rule, T=5400, float32: output cosine 1.0000207 (max |diff| 2.9e-07)
   gradient of q   : cosine 1.0000199, max |diff| 1.6e-06 (largest 1.04)
   gradient of k   : cosine 1.0000249, max |diff| 1.7e-06 (largest 1.11)
   gradient of v   : cosine 1.0000250, max |diff| 2.6e-07 (largest 0.10)
   gradient of g   : cosine 1.0000000, max |diff| 8.6e-07 (largest 0.67)
   gradient of beta: cosine 0.9999999, max |diff| 1.1e-06 (largest 0.86)
   forward + backward: PyTorch MPS 0.072s, MLX 0.283s (best of 3)
2. the inverse's gradient against float64: cosine 1.00000000, max |diff| 1.5e-05
3. the policy (07's seed), recorded episodes:
    3891 tokens ( 756 the policy's): log-prob sums   -55.50 /   -55.14, per token mean |diff| 0.0022 (max 0.310) | gradient cosine median 0.9858 (min 0.9464), norm ratio 0.962 | forward + backward 4.9s / 5.3s
    4341 tokens (1136 the policy's): log-prob sums   -38.15 /   -38.22, per token mean |diff| 0.0012 (max 0.125)

In [3]:
# 5. train_ppo_mlx's bookkeeping against 08's PyTorch functions (08.5_mlx_stack.py bookkeeping-check): two batches of
# 8 recorded episodes with random values and rewards, the critic's heads on a synthetic window of PPO's size
BK = jload(HERE / "08.5_mlx_stack_bookkeeping_checks.json")
if BK is None:
    print("bookkeeping-check not run yet")
else:
    print(f"5. the bookkeeping, MLX against PyTorch ({BK['episodes']} recorded episodes): largest absolute difference")
    for k, v in BK["max_abs_diff"].items():
        print(f"   {k:34} {v:.1e}")

5. the bookkeeping, MLX against PyTorch (16 recorded episodes): largest absolute difference
   batch_mask                         0.0e+00
   batch_answer                       0.0e+00
   batch_turn_end                     0.0e+00
   batch_content                      0.0e+00
   batch_rewards                      0.0e+00
   batch_turn                         0.0e+00
   gae_turn_advantages                4.8e-07
   gae_turn_returns                   6.0e-07
   gae_advantages                     9.5e-07
   normalized_advantages_batch        2.4e-07
   normalized_advantages_minibatch    1.5e-08
   per_turn_tokens                    0.0e+00
   prover_progress                    0.0e+00
   separate_critic_stats              1.3e-08
   critic_fit                         0.0e+00
   critic_heads_value                 8.1e-07
   critic_heads_success               3.4e-08


## The run: D37 on MLX, against 07 and 08's v1

The same seed and 07's first 25 steps: every run sees the same questions at every step, so the sampled rewards pair by step. The evaluations' 60 dev questions are the same for all; 07's are PyTorch-generated, v1's from step 80 on and 08.5's MLX-generated.


In [4]:
def load_run(d):
    if not (d / "metrics.jsonl").exists():
        return None
    return {"m": rows_of(d / "metrics.jsonl"), "ev": rows_of(d / "eval.jsonl") if (d / "eval.jsonl").exists() else [],
            "end": jload(d / "ending.json")}


RUN = {}
for label, d in (("07, PyTorch", RUNS07 / PPO07), ("08 v1, PyTorch (MLX generation from step 80)", RUNS08 / PPO08_V1),
                 ("08.5 D37, MLX", RUNS / PPO085), ("07c+D37, MLX (07's critic, D37's details, no prover)", RUNS / PPO085_PC)):
    run = load_run(d)
    if run is None:
        print(f"{label}: not run yet")
        continue
    RUN[label.split(",")[0]] = run
    m, end = run["m"], run["end"]
    act = [r for r in m if not r["critic_warmup"]]
    own = [r for r in m if not r.get("warm_from")]  # the steps this run took itself
    print(f"{label}: {len(m)} steps" + (f"; ended at {end['stopped_at']} ({end['reason']}), best evaluation {end['best_reward']:.4f} at step {end['best_step']}"
                                        if end else " (running)"))
    if act:
        def mean(k, spec):
            xs = [r[k] for r in act if r.get(k) is not None]
            return format(sum(xs) / len(xs), spec) if xs else "-"
        print(f"   actor steps {len(act)}: reward {mean('reward_mean', '.3f')} decision {mean('decision_correct', '.3f')} evidence {mean('evidence', '.3f')} "
              f"| recovered {mean('rec_recovered_given_error', '.3f')} | kl {mean('kl_mean', '.4f')} | approx_kl max {mean('approx_kl_max', '.4f')} "
              f"| first ratio {mean('ratio_first', '.4f')} | clip frac {mean('clip_frac', '.3f')} | optimizer steps {mean('updates', '.0f')}")
    for g in ("torch", "mlx", "mlx-batch"):  # who generated the step's rollouts; the rest of the step is the trainer's (and the critic's)
        rows = [r for r in own if not r["critic_warmup"] and r.get("generator", "torch") == g]
        if rows:
            print(f"   generated by {g:5}: {len(rows)} steps, median step {median([r['step_seconds'] for r in rows]):.0f}s = generation "
                  f"{median([r['gen_seconds'] for r in rows]):.0f}s + the rest {median([r['step_seconds'] - r['gen_seconds'] for r in rows]):.0f}s")
    for r in run["ev"]:
        g = r.get("recovery", {}).get("recovered_given_error")
        print(f"   eval @{r['step']:>4}: reward {r['reward']:.4f} decision {r['decision_correct']:.3f} evidence {r['evidence']:.3f} "
              f"| recovered {'-' if g is None else f'{g:.3f}'}" + ("  (07's)" if r.get("reused") else ""))


def sampled(run, lo, hi, k="reward_mean"):
    xs = [r[k] for r in run["m"] if not r["critic_warmup"] and lo <= r["step"] <= hi and r.get(k) is not None]
    return sum(xs) / len(xs) if xs else None


def paired_by_step(A, B, lo=0, hi=10 ** 9):
    """B's sampled reward minus A's over the actor steps both ran in [lo, hi]: the same questions at every step."""
    a = {r["step"]: r["reward_mean"] for r in A["m"] if not r["critic_warmup"] and lo <= r["step"] <= hi}
    b = {r["step"]: r["reward_mean"] for r in B["m"] if not r["critic_warmup"] and lo <= r["step"] <= hi}
    d = [b[s] - a[s] for s in sorted(set(a) & set(b))]
    mu = sum(d) / len(d)
    sd = (sum((x - mu) ** 2 for x in d) / max(1, len(d) - 1)) ** 0.5
    return len(d), mu, mu - 1.96 * sd / len(d) ** 0.5, mu + 1.96 * sd / len(d) ** 0.5


print("\nsampled reward by 25 steps (8 episodes a step, the same questions in every run)")
print("  steps     " + " ".join(f"{k:>8}" for k in RUN))
for lo in range(25, max(r["step"] for run in RUN.values() for r in run["m"]) + 1, 25):
    print(f"  {lo:3d}-{lo + 24:3d}   " + " ".join(f"{'-' if sampled(r, lo, lo + 24) is None else format(sampled(r, lo, lo + 24), '.3f'):>8}" for r in RUN.values()))
names = list(RUN)
for i, la in enumerate(names):
    for lb in names[i + 1:]:
        got = paired_by_step(RUN[la], RUN[lb])
        if got[0] > 1:
            n, mu, lo_, hi_ = got
            print(f"{lb} - {la}, paired by step over the {n} actor steps both ran: {mu:+.4f} (95% {lo_:+.4f} to {hi_:+.4f})")

07, PyTorch: 120 steps; ended at 120 (step budget 120 reached), best evaluation 0.6730 at step 120
   actor steps 95: reward 0.538 decision 0.516 evidence 0.467 | recovered 0.452 | kl 0.0019 | approx_kl max - | first ratio - | clip frac 0.001 | optimizer steps -
   generated by torch: 95 steps, median step 474s = generation 148s + the rest 318s
   eval @   0: reward 0.5856 decision 0.550 evidence 0.500 | recovered 0.400  (07's)
   eval @  40: reward 0.6580 decision 0.633 evidence 0.600 | recovered 0.448
   eval @  80: reward 0.6479 decision 0.633 evidence 0.583 | recovered 0.438
   eval @ 120: reward 0.6730 decision 0.683 evidence 0.633 | recovered 0.500
08 v1, PyTorch (MLX generation from step 80): 120 steps; ended at 120 (step budget 120 reached), best evaluation 0.6792 at step 40
   actor steps 95: reward 0.528 decision 0.499 evidence 0.443 | recovered 0.398 | kl 0.0017 | approx_kl max 0.0002 | first ratio 1.0000 | clip frac 0.001 | optimizer steps 4
   generated by torch: 55 steps,

08.5 D37, MLX: 380 steps; ended at 380 (converged: 6 evaluations without beating 0.7865 (step 260)), best evaluation 0.7865 at step 260
   actor steps 355: reward 0.562 decision 0.542 evidence 0.490 | recovered 0.341 | kl 0.0094 | approx_kl max 0.0007 | first ratio 1.0001 | clip frac 0.002 | optimizer steps 16
   generated by mlx  : 240 steps, median step 357s = generation 57s + the rest 298s
   generated by mlx-batch: 115 steps, median step 314s = generation 23s + the rest 293s
   eval @   0: reward 0.5856 decision 0.550 evidence 0.500 | recovered 0.400  (07's)
   eval @  40: reward 0.5693 decision 0.567 evidence 0.450 | recovered 0.400
   eval @  80: reward 0.6647 decision 0.683 evidence 0.633 | recovered 0.667
   eval @ 120: reward 0.6866 decision 0.750 evidence 0.633 | recovered 0.667
   eval @ 160: reward 0.7457 decision 0.817 evidence 0.717 | recovered 0.708
   eval @ 200: reward 0.7709 decision 0.817 evidence 0.717 | recovered 0.600
   eval @ 220: reward 0.7211 decision 0.817 ev

07c+D37, MLX (07's critic, D37's details, no prover): 286 steps (running)
   actor steps 261: reward 0.571 decision 0.528 evidence 0.512 | recovered 0.321 | kl 0.0086 | approx_kl max 0.0006 | first ratio 1.0001 | clip frac 0.002 | optimizer steps 16
   generated by mlx-batch: 261 steps, median step 304s = generation 24s + the rest 279s
   eval @   0: reward 0.5856 decision 0.550 evidence 0.500 | recovered 0.400  (07's)
   eval @  40: reward 0.5984 decision 0.583 evidence 0.517 | recovered 0.444
   eval @  60: reward 0.6945 decision 0.700 evidence 0.683 | recovered 0.586
   eval @  80: reward 0.6595 decision 0.650 evidence 0.617 | recovered 0.409
   eval @ 100: reward 0.6841 decision 0.667 evidence 0.650 | recovered 0.400
   eval @ 120: reward 0.7083 decision 0.667 evidence 0.667 | recovered 0.438
   eval @ 140: reward 0.7564 decision 0.800 evidence 0.700 | recovered 0.857
   eval @ 160: reward 0.6804 decision 0.717 evidence 0.633 | recovered 0.375
   eval @ 180: reward 0.7238 decision 

## Held out

08's probes, all generated by MLX (`08.5_mlx_stack_eval_heldout.sh`, no PyTorch in the process): greedy on the same 120 test questions and all 120 of holdout_shift, the forced probe (greedy, the 144 dev questions with a slip, each started from its slipped turns) and the 120 dev questions sampled once at T=1. D37 at step 200 (its best evaluation and its last) and at step 120 (07's and v1's last step); the seed's, 07's and v1's probes are 08's (`08_prover_critic_runs/`). Paired by question.

In [5]:
import random
TEST120 = {c["id"] for c in random.Random(0).sample(pc.load_split("test"), 120)}  # probe --limit 120's questions
MLXP = RUNS08 / "mlx-probes"  # 07's seed and PPO checkpoint through 08's probe, generated by MLX


def summary_of(d, tag, ids=None):
    """A probe's measures from its episodes (on the questions in ``ids`` when given); None until the probe is done."""
    if not (d / f"{tag}.json").exists():
        return None
    rows = [r for r in rows_of(d / f"{tag}.episodes.jsonl") if ids is None or r["id"] in ids]
    sc = [r["score"] for r in rows]
    n = len(rows)
    return {"n": n, "reward": sum(x["total"] for x in sc) / n, "decision_correct": sum(x["decision_correct"] for x in sc) / n,
            "evidence": sum(x["evidence"] for x in sc) / n,
            "answer_found": sum(pc.extract_answer(r["final_text"])[0] is not None for r in rows) / n,
            "recovery": pc.recovery_block([pc.recovery_stats(r["calls"]) for r in rows], sc)}


def recov_line(o):
    r = o["recovery"]
    f = lambda x: "  -  " if x is None else f"{x:.3f}"  # noqa: E731
    return (f"n {o['n']:3d} reward {o['reward']:.3f} decision {o['decision_correct']:.3f} evidence {o['evidence']:.3f} "
            f"answers {o['answer_found']:.3f} | erred {r['sim_erred']:.3f} recovered {f(r['recovered_given_error'])} "
            f"decision after an error {f(r['decision_given_error'])} | repeats/ep {r['repeats_per_episode']:.2f}")


def paired(d1, d2, tag, ids=None):
    """d1's reward minus d2's, question by question: the mean, its standard error, the questions won and lost."""
    if not ((d1 / f"{tag}.json").exists() and (d2 / f"{tag}.json").exists()):
        return None
    a = {r["id"]: r["score"]["total"] for r in rows_of(d1 / f"{tag}.episodes.jsonl")}
    b = {r["id"]: r["score"]["total"] for r in rows_of(d2 / f"{tag}.episodes.jsonl")}
    d = [a[i] - b[i] for i in sorted(set(a) & set(b)) if ids is None or i in ids]
    mu = sum(d) / len(d)
    return mu, (sum((x - mu) ** 2 for x in d) / len(d)) ** 0.5 / len(d) ** 0.5, sum(x > 1e-9 for x in d), sum(x < -1e-9 for x in d)


def probe_dirs(run_dir, tag):
    return [(f"{tag} {d.name.replace('probes-', '')}", d)
            for d in sorted(run_dir.glob("probes-step*"), key=lambda p: int(p.name.split("step")[1]))]


REFS = [("seed (07's)", MLXP / "seed-n200-r100"), ("07 step120", MLXP / f"07-{PPO07}-step120")] + probe_dirs(RUNS08 / PPO08_V1, "v1")
OURS = probe_dirs(RUNS / PPO085, "D37") + probe_dirs(RUNS / PPO085_PC, "07c+D37")
PROBES = (("test", TEST120, "test-T0-n120-stop", "test"), ("holdout_shift", None, "holdout_shift-T0-stop", "holdout_shift"),
          ("dev, forced (greedy)", None, "dev-T0-stop-forced", "forced"), ("dev, 120 at T=1", None, "dev-T1-n120-stop", "T=1"))
for split, ids, tag, _ in PROBES:
    print(f"{split}:")
    for label, d in REFS + OURS:
        o = summary_of(d, tag, ids)
        print(f"  {label:12} " + (recov_line(o) if o else "(not run yet)"))
fmt_p = lambda p: "-" if p is None else f"{p[0]:+.3f} ({p[1]:.3f}) {p[2]}/{p[3]}"  # noqa: E731
print("\nreward, D37 minus each, paired by question: the difference (its standard error), questions won/lost")
print(f"  {'':25} " + " ".join(f"{short:>22}" for *_, short in PROBES))
for label, d in OURS:
    for ref, dref in REFS:
        print(f"  {label:10} - {ref:12} " + " ".join(f"{fmt_p(paired(d, dref, tag, ids)):>22}" for _, ids, tag, _ in PROBES))
ours = dict(OURS)
if "D37 step200" in ours:  # against the end of D37's first cycle (step 200): before it, and the continuation after
    for label, d in OURS:
        if label != "D37 step200":
            print(f"  {label:10} - {'D37 step200':12} " + " ".join(f"{fmt_p(paired(d, ours['D37 step200'], tag, ids)):>22}" for _, ids, tag, _ in PROBES))

test:
  seed (07's)  n 120 reward 0.564 decision 0.525 evidence 0.475 answers 0.700 | erred 0.483 recovered 0.431 decision after an error 0.276 | repeats/ep 0.31
  07 step120   n 120 reward 0.630 decision 0.592 evidence 0.575 answers 0.708 | erred 0.508 recovered 0.541 decision after an error 0.295 | repeats/ep 0.42
  v1 step40    n 120 reward 0.654 decision 0.608 evidence 0.617 answers 0.792 | erred 0.333 recovered 0.475 decision after an error 0.250 | repeats/ep 0.24
  v1 step120   n 120 reward 0.612 decision 0.575 evidence 0.558 answers 0.700 | erred 0.475 recovered 0.404 decision after an error 0.228 | repeats/ep 0.39
  D37 step120  n 120 reward 0.647 decision 0.683 evidence 0.592 answers 0.775 | erred 0.392 recovered 0.426 decision after an error 0.298 | repeats/ep 0.24
  D37 step200  n 120 reward 0.721 decision 0.742 evidence 0.675 answers 0.825 | erred 0.308 recovered 0.459 decision after an error 0.324 | repeats/ep 0.17
  D37 step260  n 120 reward 0.635 decision 0.675 evidence 

  D37 step380  n 120 reward 0.594 decision 0.508 evidence 0.542 answers 0.825 | erred 0.292 recovered 0.486 decision after an error 0.171 | repeats/ep 0.35
  07c+D37 step120 n 120 reward 0.580 decision 0.475 evidence 0.542 answers 0.742 | erred 0.325 recovered 0.231 decision after an error 0.103 | repeats/ep 0.58
dev, forced (greedy):
  seed (07's)  n 144 reward 0.688 decision 0.639 evidence 0.757 answers 0.764 | erred 1.000 recovered 0.812 decision after an error 0.639 | repeats/ep 0.69
  07 step120   n 144 reward 0.760 decision 0.771 evidence 0.785 answers 0.833 | erred 1.000 recovered 0.833 decision after an error 0.771 | repeats/ep 0.38
  v1 step40    n 144 reward 0.696 decision 0.660 evidence 0.750 answers 0.757 | erred 1.000 recovered 0.799 decision after an error 0.660 | repeats/ep 0.81
  v1 step120   n 144 reward 0.700 decision 0.681 evidence 0.715 answers 0.785 | erred 1.000 recovered 0.785 decision after an error 0.681 | repeats/ep 0.50
  D37 step120  n 144 reward 0.692 decis

  D37 step260  n 144 reward 0.684 decision 0.646 evidence 0.729 answers 0.757 | erred 1.000 recovered 0.764 decision after an error 0.646 | repeats/ep 0.69
  D37 step380  n 144 reward 0.675 decision 0.611 evidence 0.736 answers 0.757 | erred 1.000 recovered 0.778 decision after an error 0.611 | repeats/ep 0.62
  07c+D37 step120 n 144 reward 0.725 decision 0.694 evidence 0.778 answers 0.792 | erred 1.000 recovered 0.826 decision after an error 0.694 | repeats/ep 0.86
dev, 120 at T=1:
  seed (07's)  n 120 reward 0.496 decision 0.442 evidence 0.375 answers 0.667 | erred 0.592 recovered 0.423 decision after an error 0.239 | repeats/ep 0.50
  07 step120   n 120 reward 0.537 decision 0.517 evidence 0.458 answers 0.633 | erred 0.608 recovered 0.479 decision after an error 0.329 | repeats/ep 0.35
  v1 step40    n 120 reward 0.484 decision 0.425 evidence 0.383 answers 0.642 | erred 0.550 recovered 0.394 decision after an error 0.242 | repeats/ep 0.51
  v1 step120   n 120 reward 0.508 decision 0

  D37 step120  n 120 reward 0.538 decision 0.558 evidence 0.483 answers 0.708 | erred 0.500 recovered 0.450 decision after an error 0.283 | repeats/ep 0.38
  D37 step200  n 120 reward 0.606 decision 0.625 evidence 0.508 answers 0.767 | erred 0.433 recovered 0.519 decision after an error 0.308 | repeats/ep 0.34
  D37 step260  n 120 reward 0.580 decision 0.600 evidence 0.467 answers 0.783 | erred 0.317 recovered 0.421 decision after an error 0.211 | repeats/ep 0.17
  D37 step380  n 120 reward 0.597 decision 0.592 evidence 0.517 answers 0.783 | erred 0.350 recovered 0.500 decision after an error 0.310 | repeats/ep 0.28
  07c+D37 step120 n 120 reward 0.530 decision 0.508 evidence 0.475 answers 0.683 | erred 0.467 recovered 0.286 decision after an error 0.179 | repeats/ep 1.06

reward, D37 minus each, paired by question: the difference (its standard error), questions won/lost
                                              test          holdout_shift                 forced                    

  D37 step120 - v1 step40      -0.007 (0.032) 36/35   +0.027 (0.032) 49/41   -0.003 (0.017) 17/31   +0.053 (0.034) 53/51
  D37 step120 - v1 step120     +0.035 (0.024) 40/30   -0.028 (0.027) 35/40   -0.007 (0.018) 15/43   +0.029 (0.029) 49/43
  D37 step200 - seed (07's)    +0.157 (0.034) 67/22   +0.061 (0.028) 58/27   +0.021 (0.020) 22/22   +0.110 (0.031) 65/34


...

  D37 step200 - v1 step120     +0.109 (0.027) 48/23   -0.037 (0.023) 35/38   +0.010 (0.018) 14/30   +0.098 (0.030) 63/34
  D37 step260 - seed (07's)    +0.070 (0.038) 54/40   -0.015 (0.030) 52/46   -0.004 (0.022) 24/28   +0.084 (0.037) 61/41
  D37 step260 - 07 step120     +0.005 (0.031) 42/46   -0.087 (0.028) 44/53   -0.077 (0.022) 15/35   +0.043 (0.036) 55/47
  D37 step260 - v1 step40      -0.019 (0.037) 39/52   -0.058 (0.031) 44/57   -0.012 (0.023) 22/29   +0.096 (0.037) 61/41
  D37 step260 - v1 step120     +0.022 (0.032) 45/43   -0.113 (0.026) 35/54   -0.016 (0.021) 18/34   +0.072 (0.036) 64/39


  D37 step380 - seed (07's)    +0.133 (0.038) 64/29   +0.078 (0.027) 58/24   -0.013 (0.023) 24/31   +0.101 (0.036) 58/35
  D37 step380 - 07 step120     +0.068 (0.033) 53/32   +0.006 (0.027) 55/33   -0.085 (0.023) 15/42   +0.060 (0.037) 55/43
  D37 step380 - v1 step40      +0.044 (0.036) 46/38   +0.036 (0.025) 48/34   -0.020 (0.024) 21/34   +0.112 (0.038) 70/35


  D37 step380 - v1 step120     +0.085 (0.033) 52/27   -0.019 (0.024) 49/33   -0.024 (0.022) 20/43   +0.088 (0.035) 63/30


  07c+D37 step120 - seed (07's)    +0.081 (0.031) 47/30   +0.064 (0.025) 54/31   +0.037 (0.020) 23/14   +0.034 (0.034) 56/43
  07c+D37 step120 - 07 step120     +0.015 (0.026) 35/34   -0.008 (0.026) 43/37   -0.035 (0.021) 16/25   -0.007 (0.036) 49/53
  07c+D37 step120 - v1 step40      -0.009 (0.034) 30/40   +0.022 (0.025) 49/37   +0.029 (0.020) 16/14   +0.046 (0.038) 56/49
  07c+D37 step120 - v1 step120     +0.033 (0.027) 41/30   -0.033 (0.023) 35/39   +0.025 (0.023) 21/22   +0.022 (0.035) 52/48
  D37 step120 - D37 step200    -0.074 (0.024) 15/38   +0.009 (0.024) 30/30   -0.017 (0.013) 10/19   -0.069 (0.027) 34/55


  D37 step260 - D37 step200    -0.086 (0.029) 22/50   -0.076 (0.027) 29/54   -0.026 (0.022) 21/23   -0.026 (0.031) 39/54
  D37 step380 - D37 step200    -0.023 (0.030) 29/34   +0.017 (0.025) 39/28   -0.034 (0.021) 17/29   -0.010 (0.029) 44/39
  07c+D37 step120 - D37 step200    -0.076 (0.025) 22/38   +0.003 (0.023) 35/40   +0.015 (0.023) 28/18   -0.076 (0.029) 37/55


## Summary

1. **Same numbers.** Against 08's PyTorch pieces: the rule to float32's rounding (gradients to 2e-6), the inverse's gradient to float64's; the policy's log-probabilities to bf16's (per token mean |diff| 0.0005-0.002), its LoRA gradients at a median cosine of 0.986 (norm ratio 0.96-1.02); the critic's features at 0.9998; the bookkeeping to 1e-6, the masks identical. The run printed "PyTorch loaded in this process: False" at both of its ends.
2. **Faster.** A step 352 s (median of steps 25-199: generation 57 s + the rest 295 s), against 452 s for v1 with MLX generating and PyTorch training (53 + 390 s) and 554 s all in PyTorch (166 + 382 s): 22% and 36% less. The rest -- the update, the reference's log-probabilities, the critic's reads -- went 390 → 295 s on the same tokens (D37's 16 optimizer steps are 4 epochs over 4 minibatches, v1's 4 over one: 4 passes over the batch each). Batched decoding (`rollout_mlx_batched`, from step 265): generation 22.8 s a step against 57.5 s (medians of steps 265-379 and 200-264), a step 314 s against 368 s; at T=0 on the same work 3.5-4.7× faster, 26 of 32 episodes token for token `rollout_mlx`'s, the other 6 split at bf16 ties of 0-1 ulp.
3. **Learning.** Greedy on the 60 dev questions D37 went 0.586 → 0.569 / 0.665 / 0.687 / 0.746 / 0.771 at steps 40-200 (07 0.658 / 0.648 / 0.673 and v1 0.679 / 0.592 / 0.622 at 40 / 80 / 120). Over the 95 actor steps all three ran, though, the sampled reward, paired by step, does not separate them: D37 − 07 −0.014 (95% −0.040 to +0.013), D37 − v1 −0.004 (−0.028 to +0.020). D37's rise came after step 120: 0.548 at steps 100-124, 0.613 and 0.606 over 150-199. The update stayed small (approx_kl at most 0.001, clip fraction 0.002, the first ratio 1.0001), the KL to the seed rose 0.001 → 0.008 and was flat from step 125 as the lr annealed to 0.
4. **Held out, at the same step** (paired by question; D37's step 120 against 07's and v1's). Test +0.017 ± 0.028 and +0.035 ± 0.024, holdout_shift −0.002 ± 0.028 and −0.028 ± 0.027, dev at T=1 +0.001 ± 0.028 and +0.029 ± 0.029: no difference. The forced probe −0.068 ± 0.014 against 07 (7 questions won, 42 lost) and −0.007 ± 0.018 against v1. The three of the 37 details did not change what 120 steps learned, and D37, like v1, did not learn 07's recovery from a forced slip (on the seed +0.004 ± 0.019, 07 +0.072 ± 0.018).
5. **Held out, after 200 steps.** Steps 120-200 added +0.074 ± 0.024 on test and +0.069 ± 0.027 at T=1, nothing on holdout_shift (−0.009 ± 0.024), +0.017 ± 0.013 on the forced probe. Against 07's step 120: test +0.091 ± 0.029, T=1 +0.070 ± 0.029, holdout_shift −0.011 ± 0.027, forced −0.051 ± 0.016. D37 gained by erring less (on test a simulator error in 31% of episodes, 07 51%, the seed 48%) and deciding right (0.742 against 0.592), not by recovering. Whether 07 or v1 would gain as much from 80 more steps is not tested: both stopped at 120 (their states at step 120 are kept).
6. **Continued to convergence** (steps 200-380, the lr annealed again from 2e-6). The sampled reward fell, 0.610 over steps 150-199 → 0.556 (200-259) → 0.547 (260-319), and came back to 0.605 (320-379) as the lr went low; the KL to the seed rose 0.008 → 0.015. The evaluations read 0.721 / 0.740 / **0.787** / 0.750 / 0.739 / 0.726 / 0.714 / 0.716 / 0.718 at steps 220-380, and the run stopped at 380, six without beating step 260's.
7. **Held out after the continuation** (paired by question, against step 200). Step 260, the evaluations' best: test −0.086 ± 0.029, holdout_shift −0.076 ± 0.027, forced −0.026 ± 0.022, T=1 −0.026 ± 0.031 -- the worst of the three checkpoints on test, holdout_shift and T=1. Step 380: test −0.023 ± 0.030, holdout_shift +0.017 ± 0.025, forced −0.034 ± 0.021, T=1 −0.010 ± 0.029. The continuation added nothing held out, and the best of nine readings of 60 questions picked their noise. Step 200 stays D37's checkpoint.